# LeetCode 2883 — Drop Missing Data bằng Python (pandas)

Link đề: https://leetcode.com/problems/drop-missing-data/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def dropMissingData(students: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong bảng `students`, **xóa các hàng có `name` bị thiếu** (`None` / `NaN`), giữ nguyên các hàng còn lại.
2. Hiểu "giá trị thiếu" trong pandas là gì (`None`, `NaN`, `NA`), vì sao **không** so sánh bằng `== None` mà phải dùng `isna()` / `notna()`.
3. Đi từ cách **vòng lặp thuần Python tự lọc từng hàng** → **`dropna(subset=["name"])`** → **mặt nạ `notna()`** → **`drop(index=...)`** → **`loc` với `lambda`**.
4. Biết **nhãn index gốc được giữ** sau khi bỏ hàng (có "lỗ hổng" như `0, 2, 3`) và khi nào mới cần `reset_index(drop=True)`.
5. Tránh các lỗi kinh điển: quên `subset` (bỏ nhầm hàng thiếu ở cột khác), `== None`, `inplace=True` rồi `return` kết quả `None`, hiểu nhầm chuỗi rỗng `""` là "thiếu".

## 2. Đề bài & ví dụ

Có một bảng `students` với các cột `student_id`, `name`, `age`. Một vài học sinh **bị thiếu tên** (`name` là giá trị thiếu).

Hãy viết hàm **xóa mọi hàng có `name` bị thiếu** và trả về bảng đã lọc.

**Ví dụ**

- Input: DataFrame `students`

| student_id | name | age |
|---|---|---|
| 32 | Piper | 5 |
| 217 | None | 19 |
| 779 | Georgia | 20 |
| 849 | Willow | 14 |

- Output:

| student_id | name | age |
|---|---|---|
| 32 | Piper | 5 |
| 779 | Georgia | 20 |
| 849 | Willow | 14 |

- Giải thích: học sinh có `student_id = 217` không có tên nên bị xóa; ba học sinh còn lại giữ nguyên thứ tự.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc. Việc kết quả có đánh lại nhãn hàng hay không cũng nên đối chiếu với đề gốc; ở notebook này ta giữ nhãn gốc, đúng mặc định của `dropna`.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "student_id": [32, 217, 779, 849],
    "name": ["Piper", None, "Georgia", "Willow"],
    "age": [5, 19, 20, 14],
})
print(students)
print()
print(students.dtypes)
print("pandas", pd.__version__)

   student_id     name  age
0          32    Piper    5
1         217      NaN   19
2         779  Georgia   20
3         849   Willow   14

student_id    int64
name            str
age           int64
dtype: object
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. "Giá trị thiếu" trong pandas

Trong dữ liệu thật, có ô **không có giá trị**: học sinh chưa khai tên, cảm biến không ghi số... Pandas biểu diễn "thiếu" bằng vài loại khác nhau:

- `None`: giá trị rỗng của Python;
- `np.nan` (`NaN`, "Not a Number"): số thực đặc biệt của NumPy;
- `pd.NA`: giá trị thiếu "mới" của pandas.

Với người học, ghi nhớ một điều: **dù viết `None` hay `NaN`, pandas đều coi là thiếu**, và có hàm riêng để kiểm tra chúng là `pd.isna(...)`.

In [2]:
print(pd.isna(None))        # True
print(pd.isna(np.nan))      # True
print(pd.isna(pd.NA))       # True
print(pd.isna("Piper"))     # False: có chữ thì không thiếu
print(pd.isna(""))          # False: chuỗi rỗng KHÔNG phải giá trị thiếu
print(pd.isna(0))           # False: số 0 cũng KHÔNG phải giá trị thiếu

True
True
True
False
False
False


### 3.2. `Series.isna()` và `Series.notna()`

Một cột của DataFrame là một `Series`. `s.isna()` trả về Series `True/False` cùng độ dài: `True` ở ô **thiếu**. `s.notna()` làm ngược lại: `True` ở ô **có giá trị**. Hai kết quả luôn đối nghịch nhau.

Dòng `students["name"]` lấy ra cột `name`.

In [3]:
name = students["name"]
print(name.tolist())            # ô thứ 1 hiển thị nan (None đã được đổi thành NaN)
print(name.isna().tolist())     # [False, True, False, False]
print(name.notna().tolist())    # [True, False, True, True]
print("số ô thiếu:", name.isna().sum())   # True được đếm là 1 nên sum() = số ô thiếu

['Piper', nan, 'Georgia', 'Willow']
[False, True, False, False]
[True, False, True, True]
số ô thiếu: 1


### 3.3. Vì sao KHÔNG dùng `== None`?

Với số thực, `NaN` có tính chất lạ: **`NaN` không bằng chính nó** (`np.nan == np.nan` là `False`). Vì vậy so sánh bằng `==` để tìm ô thiếu **không bao giờ cho kết quả đúng**. Hãy xem thí nghiệm:

In [4]:
print(np.nan == np.nan)                      # False  (!)
print((students["name"] == None).tolist())   # toàn False - không tìm ra ô thiếu
print((students["name"] != None).tolist())   # toàn True  - tưởng "không thiếu" nhưng sai
print(students["name"].isna().tolist())      # [False, True, False, False]  <- cách đúng

False
[False, False, False, False]
[True, True, True, True]
[False, True, False, False]


Kết luận: **luôn dùng `isna()` / `notna()`** (hoặc `pd.isna(x)` cho một giá trị đơn lẻ), không dùng `== None`, `== np.nan`, `!= np.nan`.

### 3.4. Mặt nạ (mask) boolean và dấu `~`

Một Series `True/False` dùng làm **mặt nạ**: đặt vào trong `df[...]` thì pandas **giữ các hàng ứng với `True`**. Toán tử `~` **đảo** `True` ↔ `False` (đọc là "phủ định").

Ví dụ: `students["name"].notna()` đã là "hàng nào có tên". Ta cũng có thể viết `~students["name"].isna()` (phủ định của "thiếu"); hai cách cho kết quả giống hệt.

In [5]:
mask = students["name"].notna()
print(mask.tolist())                           # [True, False, True, True]
print((~students["name"].isna()).tolist())     # giống hệt mask
print(students[mask])                          # chỉ giữ hàng True

[True, False, True, True]
[True, False, True, True]
   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


### 3.5. `DataFrame.dropna(subset=...)`

`df.dropna()` là hàm sẵn có để **bỏ hàng chứa giá trị thiếu**. Tham số quan trọng:

- `subset=["name"]`: **chỉ xét các cột liệt kê** (ở đây chỉ cột `name`). Nếu **không** có `subset`, pandas xét **mọi cột** và bỏ hàng nếu **bất kỳ** ô nào thiếu;
- `how="any"` (mặc định): bỏ hàng nếu **có ít nhất một** ô thiếu trong các cột đang xét; `how="all"`: chỉ bỏ nếu **tất cả** ô đang xét đều thiếu;
- `axis=0` (mặc định): bỏ **hàng**; `axis=1` sẽ bỏ **cột** (ít gặp);
- hàm **trả về bảng mới**, không sửa bảng cũ (mặc định).

In [6]:
print(students.dropna(subset=["name"]))
print()
print("bảng gốc vẫn còn", len(students), "hàng")

   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14

bảng gốc vẫn còn 4 hàng


### 3.6. Nhãn index được giữ nguyên

Mỗi hàng của DataFrame có một **nhãn** ở cột ngoài cùng bên trái (`0, 1, 2, 3`). Khi bỏ hàng, các hàng còn lại **giữ nhãn cũ**, nên kết quả có nhãn `0, 2, 3` (thiếu `1`). Nếu muốn đánh số lại `0, 1, 2`, dùng `reset_index(drop=True)` (`drop=True` để không giữ lại cột nhãn cũ).

In [7]:
res = students.dropna(subset=["name"])
print(res.index.tolist())                          # [0, 2, 3]
print(res.reset_index(drop=True).index.tolist())   # [0, 1, 2]
print(res.reset_index())                           # không có drop=True: nhãn cũ thành cột 'index'

[0, 2, 3]
[0, 1, 2]
   index  student_id     name  age
0      0          32    Piper    5
1      2         779  Georgia   20
2      3         849   Willow   14


### 3.7. Duyệt từng hàng bằng `zip` (cho cách vòng lặp)

Để hiểu bản chất, ta tự viết vòng lặp. `zip(a, b)` ghép hai dãy cùng độ dài thành từng cặp. `students.index` là dãy nhãn, `students["name"]` là dãy tên, nên `zip(students.index, students["name"])` cho ta từng cặp `(nhãn, tên)`.

Sau khi biết nhãn nào cần giữ, `df.loc[danh_sách_nhãn]` lấy đúng các hàng đó.

In [8]:
for label, name in zip(students.index, students["name"]):
    print(label, repr(name))

print(students.loc[[0, 2, 3]])     # lấy hàng theo danh sách nhãn

0 'Piper'
1 nan
2 'Georgia'
3 'Willow'
   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy làm như một giáo viên cầm danh sách lớp: **đọc từng dòng, nếu ô tên trống thì gạch bỏ**.

| nhãn | student_id | name | tên có không? | hành động |
|---|---|---|---|---|
| 0 | 32 | Piper | có | giữ |
| 1 | 217 | (trống) | **không** | **bỏ** |
| 2 | 779 | Georgia | có | giữ |
| 3 | 849 | Willow | có | giữ |

Kết quả: giữ các nhãn `[0, 2, 3]`. Đây chính là thuật toán: **lọc các hàng thỏa điều kiện "name không thiếu"**. Mọi cách ở dưới chỉ là những cách khác nhau để viết điều kiện đó.

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

**Ý tưởng.** Duyệt từng hàng, kiểm tra tên bằng `pd.isna(name)`; nếu không thiếu thì **ghi nhãn vào danh sách `keep`**. Cuối cùng lấy `df.loc[keep]`.

Cách này giống cách giáo viên gạch tên ở mục 4, nhưng chạy bằng vòng lặp Python (chậm hơn nhiều so với vectorized khi bảng lớn).

In [9]:
def drop_missing_loop(students: pd.DataFrame) -> pd.DataFrame:
    keep = []                                              # nhãn các hàng được giữ
    for label, name in zip(students.index, students["name"]):
        if not pd.isna(name):                              # tên có giá trị
            keep.append(label)
    return students.loc[keep]


print(drop_missing_loop(students))

   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


In [10]:
# Kiểm tra nhanh: hàng bị bỏ là hàng nào?
res = drop_missing_loop(students)
print("nhãn giữ:", res.index.tolist())          # [0, 2, 3]
print("id còn lại:", res["student_id"].tolist())  # [32, 779, 849]

nhãn giữ: [0, 2, 3]
id còn lại: [32, 779, 849]


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng. Vòng lặp chạy `m` lần, mỗi lần `O(1)`: **Time `O(m)`**; danh sách `keep` tối đa `m` phần tử và bảng kết quả: **Space `O(m)`**.

**Nút thắt.** Vòng lặp Python chậm với bảng hàng triệu dòng, và ta phải tự lo mọi chi tiết (`pd.isna`, `loc`). pandas đã có sẵn thao tác này.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`students.dropna(subset=["name"])`

- `subset=["name"]`: chỉ nhìn cột `name`;
- mặc định `how="any"`: ô `name` thiếu thì bỏ cả hàng;
- thứ tự hàng và nhãn gốc được giữ.

Các cách khác cho cùng kết quả: **mặt nạ `notna()`** (Cách 3), **`drop(index=...)`** xóa theo nhãn (Cách 4), **`loc` + `lambda`** (Cách 5). Biết nhiều cách giúp bạn linh hoạt khi đề đổi nhẹ (ví dụ bỏ hàng thiếu `name` **hoặc** thiếu `age`, hoặc chỉ bỏ khi thiếu cả hai).

## 7. Các cách tiếp theo

### Cách 2 — `dropna(subset=["name"])` (cách chuẩn)

**Giải thích.** Một lời gọi, không sửa bảng đầu vào (mặc định `inplace=False`), trả về bảng mới.

In [11]:
def drop_missing_dropna(students: pd.DataFrame) -> pd.DataFrame:
    return students.dropna(subset=["name"])


print(drop_missing_dropna(students))

   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def dropMissingData(students: pd.DataFrame) -> pd.DataFrame:
    return students.dropna(subset=["name"])

**Độ phức tạp Cách 2:** **Time `O(m)`** (quét cột `name` một lần), **Space `O(m)`** (bảng kết quả là bản sao các hàng giữ lại).

### Cách 3 — Mặt nạ `notna()` rồi lọc

**Giải thích.** Tách thao tác thành hai bước nhìn thấy được: (1) tạo mask `True/False` cho "tên có giá trị", (2) `df[mask]` giữ các hàng `True`. Cách này rất hữu ích khi điều kiện phức tạp hơn (kết hợp `&`, `|` với điều kiện khác).

In [13]:
def drop_missing_mask(students: pd.DataFrame) -> pd.DataFrame:
    has_name = students["name"].notna()          # Series True/False
    return students[has_name]


print(drop_missing_mask(students))

   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def dropMissingData(students: pd.DataFrame) -> pd.DataFrame:
    return students[students["name"].notna()]

**Độ phức tạp Cách 3:** **Time `O(m)`**, **Space `O(m)`** (mask `m` phần tử bool + bảng kết quả).

### Cách 4 — `drop(index=...)`: xóa theo nhãn

**Giải thích.** Thay vì "giữ cái cần", ta "xóa cái không cần": lấy **nhãn của các hàng thiếu tên** bằng `students.index[mask_thiếu]`, rồi `drop(index=nhãn)`. Lưu ý: cách này dựa vào **nhãn**; nếu index có nhãn trùng nhau thì `drop` xóa **mọi** hàng mang nhãn đó, nên chỉ dùng khi index duy nhất.

In [15]:
def drop_missing_drop(students: pd.DataFrame) -> pd.DataFrame:
    bad_labels = students.index[students["name"].isna()]   # nhãn các hàng thiếu tên
    print("nhãn sẽ xóa:", bad_labels.tolist())
    return students.drop(index=bad_labels)


print(drop_missing_drop(students))

nhãn sẽ xóa: [1]
   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


**Độ phức tạp Cách 4:** **Time `O(m)`**, **Space `O(m)`**.

### Cách 5 — `loc` kết hợp `lambda` (hợp với method chaining)

**Giải thích.** `df.loc[lambda d: d["name"].notna()]` truyền một hàm nhận chính bảng hiện tại `d` rồi trả mask. Ưu điểm: khi bạn nối nhiều bước (`.assign(...).loc[lambda d: ...]`), `d` luôn là bảng ở bước trước, không phải bảng gốc.

In [16]:
def drop_missing_loc(students: pd.DataFrame) -> pd.DataFrame:
    return students.loc[lambda d: d["name"].notna()]


print(drop_missing_loc(students))

   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m)`**.

> Ba cách 2, 3, 5 đều ngắn gọn và đều được nhận trên LeetCode. Cách chuẩn nhất, dễ đọc nhất: `dropna(subset=["name"])`.

## 8. TRACE từng bước

Ta theo dõi cả hai kiểu: (a) vòng lặp tự viết, in từng hàng; (b) pipeline pandas `isna → mask → lọc`.

In [17]:
def trace_loop(students):
    keep = []
    for step, (label, sid, name) in enumerate(zip(students.index, students["student_id"], students["name"]), start=1):
        missing = pd.isna(name)
        action = "BỎ" if missing else "giữ"
        print(f"bước {step}: nhãn={label}, student_id={sid}, name={name!r}, thiếu={missing} -> {action}")
        if not missing:
            keep.append(label)
        print(f"          keep = {keep}")
    result = students.loc[keep]
    print("\nkết quả:", result.index.tolist(), "-> số hàng", len(result))
    return result


_ = trace_loop(students)

bước 1: nhãn=0, student_id=32, name='Piper', thiếu=False -> giữ
          keep = [0]
bước 2: nhãn=1, student_id=217, name=nan, thiếu=True -> BỎ
          keep = [0]
bước 3: nhãn=2, student_id=779, name='Georgia', thiếu=False -> giữ
          keep = [0, 2]
bước 4: nhãn=3, student_id=849, name='Willow', thiếu=False -> giữ
          keep = [0, 2, 3]

kết quả: [0, 2, 3] -> số hàng 3


In [18]:
def trace_pandas(students):
    print("BƯỚC 0 - bảng gốc", students.shape)
    print(students)

    missing = students["name"].isna()
    print("\nBƯỚC 1 - isna() trên cột name:")
    print(pd.DataFrame({"name": students["name"], "isna": missing}))

    keep_mask = ~missing
    print("\nBƯỚC 2 - ~isna() (True = giữ):", keep_mask.tolist())

    result = students[keep_mask]
    print("\nBƯỚC 3 - students[mask]:", result.shape)
    print(result)
    print("\nnhãn còn lại:", result.index.tolist(), "(thiếu nhãn 1)")

    same = result.equals(students.dropna(subset=["name"]))
    print("giống dropna(subset=['name'])?", same)
    return result


_ = trace_pandas(students)

BƯỚC 0 - bảng gốc (4, 3)
   student_id     name  age
0          32    Piper    5
1         217      NaN   19
2         779  Georgia   20
3         849   Willow   14

BƯỚC 1 - isna() trên cột name:
      name   isna
0    Piper  False
1      NaN   True
2  Georgia  False
3   Willow  False

BƯỚC 2 - ~isna() (True = giữ): [True, False, True, True]

BƯỚC 3 - students[mask]: (3, 3)
   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14

nhãn còn lại: [0, 2, 3] (thiếu nhãn 1)
giống dropna(subset=['name'])? True


Bảng tóm tắt hình dạng:

| bước | đối tượng | hình dạng |
|---|---|---|
| 0 | bảng gốc | `(4, 3)` |
| 1 | `isna()` của cột `name` | `Series` 4 phần tử `[F, T, F, F]` |
| 2 | `~isna()` | `Series` 4 phần tử `[T, F, T, T]` |
| 3 | `students[mask]` | `(3, 3)`, nhãn `[0, 2, 3]` |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [19]:
import pandas as pd

def dropMissingData(students: pd.DataFrame) -> pd.DataFrame:
    return students.dropna(subset=["name"])


print(dropMissingData(students))

   student_id     name  age
0          32    Piper    5
2         779  Georgia   20
3         849   Willow   14


## 10. Vì sao thuật toán đúng

- **Đúng tiêu chí bỏ hàng:** `dropna(subset=["name"])` bỏ hàng khi và chỉ khi ô `name` của hàng đó thiếu (`None`/`NaN`). Đây đúng điều đề yêu cầu.
- **Không bỏ nhầm:** thiếu ở cột khác (`age`, `student_id`) không ảnh hưởng vì ta giới hạn bằng `subset`.
- **Chuỗi rỗng không bị bỏ:** `""` không phải giá trị thiếu theo pandas, nên hàng có `name = ""` vẫn được giữ (đề nói về giá trị thiếu, không phải chuỗi rỗng).
- **Thứ tự và nhãn:** thao tác chỉ **xóa** hàng, không sắp xếp lại; thứ tự tương đối, nhãn gốc và mọi cột còn lại giữ nguyên.
- **Tính chất kết quả:** mọi hàng trong kết quả đều có `name` không thiếu, và số hàng kết quả = số hàng gốc − số ô `name` thiếu.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột.

- **Time:** `O(m)` để quét cột `name`, cộng chi phí sao chép các hàng giữ lại (tối đa `O(m·c)` ô dữ liệu). Cách vòng lặp thuần Python cũng `O(m)` nhưng hằng số lớn hơn nhiều.
- **Space:** `O(m·c)` cho bảng kết quả (bản sao) + `O(m)` cho mask. Trong thực tế pandas có thể dùng cơ chế copy-on-write nên đôi khi không sao chép ngay, nhưng khi phân tích ta coi như tạo bảng mới.
- Việc "tìm hàng thiếu" **không thể nhanh hơn `O(m)`**, vì mỗi ô phải được xem ít nhất một lần.

## 12. Bộ test

Ta đưa các cách vào một từ điển `impls` để chạy chung mọi test.

In [20]:
impls = {
    "loop": drop_missing_loop,
    "dropna": drop_missing_dropna,
    "mask": drop_missing_mask,
    "drop": drop_missing_drop,
    "loc": drop_missing_loc,
    "submit": dropMissingData,
}

expected_example = pd.DataFrame({
    "student_id": [32, 779, 849],
    "name": ["Piper", "Georgia", "Willow"],
    "age": [5, 20, 14],
}, index=[0, 2, 3])

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(students), expected_example, check_index_type=False)
print("PASS ví dụ của đề (mọi cách)")

nhãn sẽ xóa: [1]
PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def make(names, ages=None):
    n = len(names)
    return pd.DataFrame({
        "student_id": list(range(1, n + 1)),
        "name": names,
        "age": ages if ages is not None else [10] * n,
    })


def check_all(df, expected_ids, label):
    for key, fn in impls.items():
        got = fn(df)["student_id"].tolist()
        assert got == expected_ids, (label, key, got, expected_ids)
    print("PASS", label, "->", expected_ids)


check_all(make(["a", "b", "c"]), [1, 2, 3], "không thiếu gì")
check_all(make([None, None, None]), [], "thiếu toàn bộ")
check_all(make([None, "b", "c"]), [2, 3], "thiếu ở hàng đầu")
check_all(make(["a", "b", None]), [1, 2], "thiếu ở hàng cuối")
check_all(make(["a", "", "c"]), [1, 2, 3], "chuỗi rỗng không phải thiếu")
check_all(make(["only"]), [1], "một hàng đủ tên")
check_all(make([None]), [], "một hàng thiếu tên")

nhãn sẽ xóa: []
PASS không thiếu gì -> [1, 2, 3]
nhãn sẽ xóa: [0, 1, 2]
PASS thiếu toàn bộ -> []
nhãn sẽ xóa: [0]
PASS thiếu ở hàng đầu -> [2, 3]
nhãn sẽ xóa: [2]
PASS thiếu ở hàng cuối -> [1, 2]
nhãn sẽ xóa: []
PASS chuỗi rỗng không phải thiếu -> [1, 2, 3]
nhãn sẽ xóa: []
PASS một hàng đủ tên -> [1]
nhãn sẽ xóa: [0]
PASS một hàng thiếu tên -> []


### 12.2. Tính chất của kết quả, đầu vào không bị sửa, và phản ví dụ

In [22]:
# Thiếu ở cột khác (age) thì KHÔNG được bỏ
df = make(["a", "b", "c"], ages=[1.0, np.nan, 3.0])
for key, fn in impls.items():
    assert fn(df)["student_id"].tolist() == [1, 2, 3], key
print("PASS thiếu ở cột age nhưng có name -> vẫn giữ")

# Đầu vào không bị sửa
before = students.copy()
for key, fn in impls.items():
    fn(students)
pd.testing.assert_frame_equal(students, before)
print("PASS đầu vào không bị sửa")

# Kết quả không còn name thiếu và số hàng đúng
res = dropMissingData(students)
assert res["name"].notna().all()
assert len(res) == len(students) - students["name"].isna().sum()
print("PASS tính chất kết quả")

# Phản ví dụ: bỏ subset thì bỏ nhầm hàng thiếu ở cột khác
wrong = df.dropna()
assert wrong["student_id"].tolist() == [1, 3]          # hàng 2 bị bỏ oan
print("PASS phản ví dụ: dropna() không subset bỏ nhầm ->", wrong["student_id"].tolist())

nhãn sẽ xóa: []
PASS thiếu ở cột age nhưng có name -> vẫn giữ
nhãn sẽ xóa: [1]
PASS đầu vào không bị sửa
PASS tính chất kết quả
PASS phản ví dụ: dropna() không subset bỏ nhầm -> [1, 3]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh bảng ngẫu nhiên có tỉ lệ ô `name` thiếu thay đổi, rồi so mọi cách với `dropna` (bảng chuẩn).

In [23]:
import random

random.seed(2883)
for trial in range(200):
    n = random.randint(0, 25)
    p_missing = random.choice([0.0, 0.2, 0.5, 1.0])
    names = [None if random.random() < p_missing else random.choice(["An", "Binh", "Chi", ""]) for _ in range(n)]
    ages = [random.choice([1.0, 2.0, np.nan]) for _ in range(n)]
    df = pd.DataFrame({"student_id": list(range(n)), "name": pd.Series(names, dtype="object"), "age": ages})
    ref = df.dropna(subset=["name"])
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df), ref, check_index_type=False)
print("PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau")

nhãn sẽ xóa: [11]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
nhãn sẽ xóa: [1, 4, 5, 7, 10]
nhãn sẽ xóa: [0, 1, 5, 8, 9, 12]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4]
nhãn sẽ xóa: []
nhãn sẽ xóa: [3, 9, 10, 11, 16, 17, 20, 23]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 5, 7]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7]
nhãn sẽ xóa: [10, 17, 18]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 7, 9, 15, 16]
nhãn sẽ xóa: []
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 2, 4, 6, 8, 9, 11, 14, 15]
nhãn sẽ xóa: [0, 6, 7, 10, 13, 14]
nhãn sẽ xóa: [0, 3, 6, 7, 8, 11, 12]
nhãn sẽ xóa: []
nhãn sẽ xóa: [1]
nhãn sẽ xóa: []
nhãn sẽ xóa: [6, 7, 8, 11]
nhãn sẽ xóa: [3, 4, 5, 7, 8, 9, 11, 12, 14, 15]
nhãn sẽ xóa: []
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
nhãn sẽ xóa: [1, 2, 

nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
nhãn sẽ xóa: [1, 2, 4]
nhãn sẽ xóa: [1, 13]
nhãn sẽ xóa: [6, 7, 8, 9, 10, 11, 12, 17, 19, 21, 22, 23]
nhãn sẽ xóa: [0, 2, 3, 5, 7, 8, 10, 11, 12]
nhãn sẽ xóa: []
nhãn sẽ xóa: [3, 4, 5, 10, 15]


nhãn sẽ xóa: [0, 3, 5, 7, 9, 11, 13, 16, 17, 18, 21]
nhãn sẽ xóa: []
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5]
nhãn sẽ xóa: [0, 1, 2, 3, 4]
nhãn sẽ xóa: []
nhãn sẽ xóa: [2, 3, 5, 8]
nhãn sẽ xóa: [0, 3, 4, 5, 6, 8, 9]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
nhãn sẽ xóa: [0]
nhãn sẽ xóa: [0, 1, 4, 5, 8, 11, 15, 17, 18, 19, 21, 23]
nhãn sẽ xóa: [1, 2, 4, 5, 6, 7, 10, 12, 15, 16, 17, 18, 19]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
nhãn sẽ xóa: [8]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
nhãn sẽ xóa: []
nhãn sẽ xóa: [3, 10, 13]
nhãn sẽ xóa: []
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3]
nhãn sẽ xóa: [2, 8, 12]
nhãn sẽ xóa: [3, 6]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
nhãn sẽ xóa: [0]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13

nhãn sẽ xóa: [0, 1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 16, 17, 20]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
nhãn sẽ xóa: [3, 4]
nhãn sẽ xóa: [0, 7, 14, 16, 17, 19, 21, 24]
nhãn sẽ xóa: [1, 2, 3, 5, 10, 11]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 6]
nhãn sẽ xóa: [5, 11, 12]
nhãn sẽ xóa: [2]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7]
nhãn sẽ xóa: []
nhãn sẽ xóa: [1, 4, 5, 7]
nhãn sẽ xóa: [2, 4, 9]
nhãn sẽ xóa: [1]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
nhãn sẽ xóa: []
nhãn sẽ xóa: [2, 10, 11, 14, 16, 17, 18, 21, 22, 24]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7]
nhãn sẽ xóa: []
nhãn sẽ xóa: [2, 3, 5]
nhãn sẽ xóa: [5, 8, 10, 14]
nhãn sẽ xóa: [3, 4, 8, 10]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
nhãn sẽ xóa: [

nhãn sẽ xóa: [0, 4]


nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
nhãn sẽ xóa: []
nhãn sẽ xóa: [5, 6]
nhãn sẽ xóa: [0]
nhãn sẽ xóa: [0, 5, 10, 12, 15, 18]
nhãn sẽ xóa: []
nhãn sẽ xóa: [1, 3, 5, 9, 14]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
nhãn sẽ xóa: [1, 2, 4, 6, 9, 13, 14]
nhãn sẽ xóa: []
nhãn sẽ xóa: []
nhãn sẽ xóa: [0, 1, 2, 5, 7, 9, 12, 13, 14, 16, 20, 21, 22, 23]
nhãn sẽ xóa: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
nhãn sẽ xóa: [4, 9, 14, 16, 18]
nhãn sẽ xóa: []
nhãn sẽ xóa: [12]
PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `subset` — bỏ nhầm hàng thiếu ở cột khác

In [24]:
df_other = pd.DataFrame({
    "student_id": [1, 2, 3],
    "name": ["An", "Binh", "Chi"],
    "age": [10.0, np.nan, 12.0],        # Binh thiếu tuổi nhưng có tên
})
sai = df_other.dropna()                          # xét mọi cột
print("sai  :", sai["student_id"].tolist())      # [1, 3]: Binh bị bỏ oan
dung = df_other.dropna(subset=["name"])
print("đúng :", dung["student_id"].tolist())     # [1, 2, 3]

sai  : [1, 3]
đúng : [1, 2, 3]


**Vì sao sai:** không có `subset`, `dropna` xét **mọi cột** và bỏ hàng khi **bất kỳ** ô nào thiếu. Đề chỉ yêu cầu bỏ hàng thiếu `name`. Hãy luôn ghi rõ `subset=["name"]`.

### Lỗi 2: Dùng `== None` để tìm ô thiếu

In [25]:
sai = students[students["name"] == None]     # mong tìm hàng thiếu tên
print("sai - tìm được", len(sai), "hàng (đáng lẽ 1)")
dung = students[students["name"].isna()]
print("đúng - tìm được", len(dung), "hàng:", dung["student_id"].tolist())

sai - tìm được 0 hàng (đáng lẽ 1)
đúng - tìm được 1 hàng: [217]


**Vì sao sai:** `NaN`/`None` không so sánh bằng được bằng `==` (kết quả toàn `False`). Dùng `isna()` để tìm hàng thiếu và `notna()` để tìm hàng có giá trị.

### Lỗi 3: `inplace=True` rồi `return` kết quả của nó

In [26]:
def sai_inplace(students):
    return students.dropna(subset=["name"], inplace=True)    # trả về None!


def dung_khong_inplace(students):
    return students.dropna(subset=["name"])


tmp = students.copy()
print("sai  :", sai_inplace(tmp))                       # None
print("đúng :", len(dung_khong_inplace(students)), "hàng")

sai  : None
đúng : 3 hàng


**Vì sao sai:** với `inplace=True`, hàm **sửa tại chỗ** và **trả `None`**. Hàm nộp LeetCode cần trả DataFrame, nên `return df.dropna(..., inplace=True)` trả `None` và bị chấm sai. Cách an toàn: bỏ hẳn `inplace`, `return` kết quả của `dropna`.

### Lỗi 4: Nhầm chuỗi rỗng với giá trị thiếu

In [27]:
df_empty = pd.DataFrame({"student_id": [1, 2], "name": ["An", ""], "age": [10, 11]})
print("dropna giữ lại :", len(df_empty.dropna(subset=["name"])), "hàng (chuỗi rỗng không bị bỏ)")
print("nếu đề cũng muốn bỏ chuỗi rỗng, phải lọc thêm:")
print(df_empty[df_empty["name"].notna() & (df_empty["name"] != "")])

dropna giữ lại : 2 hàng (chuỗi rỗng không bị bỏ)
nếu đề cũng muốn bỏ chuỗi rỗng, phải lọc thêm:
   student_id name  age
0           1   An   10


**Vì sao dễ nhầm:** `""` là một chuỗi hợp lệ (độ dài 0), không phải giá trị thiếu. Bài này chỉ yêu cầu bỏ giá trị **thiếu**. Nếu một đề khác yêu cầu bỏ cả chuỗi rỗng, cần điều kiện bổ sung như trên.

### Lỗi 5: Mong `reset_index` tự xảy ra

In [28]:
res = dropMissingData(students)
print("nhãn kết quả:", res.index.tolist())              # [0, 2, 3], không phải [0, 1, 2]
print("sau reset   :", res.reset_index(drop=True).index.tolist())

nhãn kết quả: [0, 2, 3]
sau reset   : [0, 1, 2]


**Lưu ý:** `dropna` **không** đánh lại nhãn. Nếu đề gốc yêu cầu index `0..k-1`, thêm `.reset_index(drop=True)`; nếu không yêu cầu, thêm vào cũng thường vô hại, nhưng hãy đối chiếu đề.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "xóa hàng thiếu / bỏ dòng trống / drop missing" → nghĩ ngay tới **`dropna`**.
- **Luôn ghi `subset=[...]`** khi chỉ quan tâm một vài cột.
- Muốn **đếm** ô thiếu mỗi cột: `df.isna().sum()`; muốn xem **hàng** nào thiếu: `df[df["name"].isna()]`.
- Muốn **thay thế** thay vì bỏ → `fillna` (bài 2887). Muốn **kiểm tra có thiếu không**: `df["name"].isna().any()`.
- `dropna` trả bảng mới: gán lại hoặc `return`. Không dùng `inplace=True` trong bài nộp.
- Chain gọn: `students.dropna(subset=["name"]).reset_index(drop=True)`.

## 15. Biến thể của bài

1. Bỏ hàng nếu thiếu `name` **hoặc** `age`: `dropna(subset=["name", "age"])` (`how="any"`).
2. Chỉ bỏ hàng nếu thiếu **cả** `name` **và** `age`: `dropna(subset=["name", "age"], how="all")`.
3. Giữ hàng có **ít nhất 2 ô đủ giá trị**: `dropna(thresh=2)`.
4. Bỏ **cột** chứa giá trị thiếu: `dropna(axis=1)`.
5. Bỏ cả chuỗi rỗng và khoảng trắng: `df[df["name"].str.strip().ne("") & df["name"].notna()]`.
6. Thay vì bỏ, **lấp** giá trị thiếu bằng `"Unknown"`: `fillna({"name": "Unknown"})`.

Ta thử vài biến thể trên dữ liệu nhỏ:

In [29]:
v = pd.DataFrame({
    "student_id": [1, 2, 3, 4],
    "name": ["An", None, "Chi", None],
    "age": [10.0, 11.0, np.nan, np.nan],
})
print("1. any  ->", v.dropna(subset=["name", "age"])["student_id"].tolist())              # [1]
print("2. all  ->", v.dropna(subset=["name", "age"], how="all")["student_id"].tolist())   # [1, 2, 3]
print("3. thresh=3 ->", v.dropna(thresh=3)["student_id"].tolist())                       # [1]
print("6. fillna ->", v.fillna({"name": "Unknown"})["name"].tolist())

1. any  -> [1]
2. all  -> [1, 2, 3]
3. thresh=3 -> [1]
6. fillna -> ['An', 'Unknown', 'Chi', 'Unknown']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp `zip` + `pd.isna` | tự gạch hàng thiếu | `O(m)` (Python, chậm) | `O(m)` | hiểu bản chất; không dùng trên dữ liệu lớn |
| 2. `dropna(subset=["name"])` | hàm có sẵn | `O(m)` | `O(m)` | **cách nộp bài** |
| 3. `df[df["name"].notna()]` | mask rồi lọc | `O(m)` | `O(m)` | muốn kết hợp thêm điều kiện `&`, `\|` |
| 4. `drop(index=nhãn_thiếu)` | xóa theo nhãn | `O(m)` | `O(m)` | khi muốn "xóa" thay vì "giữ"; index phải duy nhất |
| 5. `loc[lambda d: ...]` | mask bằng hàm | `O(m)` | `O(m)` | method chaining |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `pd.isna(None)`, `pd.isna("")`, `pd.Series([1.0, None, 3.0]).notna().tolist()`, `len(students.dropna())`, `len(students.dropna(subset=["name"]))`.

**Bài 2 (tính tay).** Với `name = ["a", None, "c", None, "e"]` (nhãn `0..4`): viết mask `notna()` và nhãn các hàng còn lại sau `dropna()`; rồi nói `dropna(how="all")` cho kết quả gì nếu bảng chỉ có cột `name`.

**Bài 3 (điền TODO).** Hoàn thành `drop_missing_todo(df)` bằng mặt nạ.

In [30]:
def drop_missing_todo(df):
    # TODO 1: has_name = df["name"].notna()
    # TODO 2: return df[has_name]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `drop_missing_loop2(students)` bằng vòng lặp thuần Python (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `drop_name_or_age(df)` bỏ hàng nếu thiếu `name` **hoặc** `age`. (b) Viết `count_missing_by_column(df)` trả về `dict` {tên cột: số ô thiếu}.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `False`, `[True, False, True]`, `3`, `3`. Bài 2: `notna()` là `True` ở ô có giá trị.
- **Mức 2 (vừa):** Bài 2: mask `[T, F, T, F, T]`, nhãn còn lại `[0, 2, 4]`; với một cột duy nhất, `how="all"` cho kết quả giống `how="any"`. Bài 3: hai dòng đã ghi trong comment. Bài 4: `keep = []`, duyệt `zip(index, name)`, nếu `not pd.isna(name)` thì thêm nhãn.
- **Mức 3 (sâu):** Bài 5(a): `df.dropna(subset=["name", "age"])`. Bài 5(b): `df.isna().sum().to_dict()` (kiểu giá trị là số nguyên NumPy; ép `int(...)` nếu cần).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert pd.isna(None) is True or pd.isna(None) == True
assert not pd.isna("")
assert pd.Series([1.0, None, 3.0]).notna().tolist() == [True, False, True]
assert len(students.dropna()) == 3
assert len(students.dropna(subset=["name"])) == 3

# Bài 2
s = pd.DataFrame({"name": ["a", None, "c", None, "e"]})
assert s["name"].notna().tolist() == [True, False, True, False, True]
assert s.dropna().index.tolist() == [0, 2, 4]
assert s.dropna(how="all").index.tolist() == [0, 2, 4]

# Bài 3
def drop_missing_todo_ans(df):
    has_name = df["name"].notna()
    return df[has_name]

pd.testing.assert_frame_equal(drop_missing_todo_ans(students), expected_example, check_index_type=False)

# Bài 4
def drop_missing_loop2(students):
    keep = []
    for label, name in zip(students.index, students["name"]):
        if not pd.isna(name):
            keep.append(label)
    return students.loc[keep]

pd.testing.assert_frame_equal(drop_missing_loop2(students), expected_example, check_index_type=False)

# Bài 5(a)
def drop_name_or_age(df):
    return df.dropna(subset=["name", "age"])

t = pd.DataFrame({"student_id": [1, 2, 3], "name": ["a", None, "c"], "age": [1.0, 2.0, np.nan]})
assert drop_name_or_age(t)["student_id"].tolist() == [1]

# Bài 5(b)
def count_missing_by_column(df):
    return {col: int(n) for col, n in df.isna().sum().items()}

assert count_missing_by_column(t) == {"student_id": 0, "name": 1, "age": 1}
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bỏ hàng thiếu ở một cột)

```python
import pandas as pd

def dropMissingData(students: pd.DataFrame) -> pd.DataFrame:
    return students.dropna(subset=["name"])
```

**Checklist**

- [ ] Có `subset=["name"]` (chỉ xét cột `name`, không bỏ oan hàng thiếu ở cột khác)?
- [ ] Dùng `isna()` / `notna()` / `dropna`, **không** `== None`?
- [ ] **Trả về** DataFrame (không `inplace=True` rồi trả `None`)?
- [ ] Biết nhãn gốc được giữ; chỉ `reset_index(drop=True)` nếu đề yêu cầu?
- [ ] Đã thử bảng không thiếu gì, toàn thiếu, thiếu ở hàng đầu/cuối, chuỗi rỗng?
- [ ] Skeleton đúng `def dropMissingData(students: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?